# Ebene 2 / Dyna: PPO in der Chronos-2-Trainingsumgebung

Chronos-2 (prediction_length=1) ersetzt die Pendulum-Dynamik, der Reward ist
analytisch. PPO + VecNormalize trainiert **nur im gelernten Modell** und wird auf
dem **echten** Pendulum-v1 evaluiert; Vergleich: PPO mit gleichem Budget direkt
auf dem echten Env, plus Random-Referenz.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import VecEnv, VecMonitor, VecNormalize

from pendulum import (
    MAX_SPEED, MAX_TORQUE, ChronosDynamics, collect_random_dataset, reward_fn, state_to_obs,
)

SEED = 0
QUICK = False          # True: 50k statt 100k Steps
TOTAL_STEPS = 50_000 if QUICK else 100_000
N_ENVS = 16
CONTEXT_LEN = 32       # informiert durch den Kontext-Sweep in Notebook 1

np.random.seed(SEED)
torch.manual_seed(SEED)

dataset = collect_random_dataset(n_episodes=50, seed=SEED)
chronos_model = ChronosDynamics()
print("Chronos:", chronos_model.model_id, "auf", chronos_model.device)

In [ ]:
class LearnedPendulumVecEnv(VecEnv):
    """Pendulum mit gelernter Dynamik; Reset = reales Kontextfenster (Dyna-Branching)."""

    def __init__(self, model, dataset, n_envs=16, context_len=32, episode_len=200, seed=0):
        self.render_mode = None
        super().__init__(
            n_envs,
            spaces.Box(np.array([-1, -1, -MAX_SPEED], dtype=np.float32),
                       np.array([1, 1, MAX_SPEED], dtype=np.float32)),
            spaces.Box(-MAX_TORQUE, MAX_TORQUE, shape=(1,), dtype=np.float32),
        )
        self.model, self.context_len, self.episode_len = model, context_len, episode_len
        self.states, self.actions = dataset["states"], dataset["actions"]
        self.rng = np.random.default_rng(seed)
        self.ctx_s = np.zeros((n_envs, context_len, 2), dtype=np.float32)
        self.ctx_a = np.zeros((n_envs, context_len, 1), dtype=np.float32)
        self.steps = np.zeros(n_envs, dtype=np.int64)

    def _reset_env(self, i):
        ep = int(self.rng.integers(len(self.actions)))
        t0 = int(self.rng.integers(1, self.actions.shape[1] - self.context_len + 2))
        self.ctx_s[i] = self.states[ep, t0 : t0 + self.context_len]
        self.ctx_a[i] = self.actions[ep, t0 - 1 : t0 - 1 + self.context_len]
        self.steps[i] = 0

    def reset(self):
        for i in range(self.num_envs):
            self._reset_env(i)
        return state_to_obs(self.ctx_s[:, -1])

    def step_async(self, actions):
        self._pending = np.clip(actions.astype(np.float32), -MAX_TORQUE, MAX_TORQUE)

    def step_wait(self):
        a = self._pending.reshape(self.num_envs, 1)
        cur = self.ctx_s[:, -1]
        rewards = reward_fn(cur[:, 0], cur[:, 1], a[:, 0]).astype(np.float32)

        nxt = self.model.predict(self.ctx_s, self.ctx_a, a[:, None, :])[:, 0]  # EIN Batch-Call
        nxt[:, 1] = np.clip(nxt[:, 1], -MAX_SPEED, MAX_SPEED)
        self.ctx_s = np.roll(self.ctx_s, -1, axis=1)
        self.ctx_a = np.roll(self.ctx_a, -1, axis=1)
        self.ctx_s[:, -1], self.ctx_a[:, -1] = nxt, a
        self.steps += 1

        dones = self.steps >= self.episode_len
        infos = [{} for _ in range(self.num_envs)]
        obs = state_to_obs(self.ctx_s[:, -1])
        for i in np.flatnonzero(dones):
            infos[i]["terminal_observation"] = obs[i].copy()
            infos[i]["TimeLimit.truncated"] = True
            self._reset_env(i)
        if dones.any():
            obs = state_to_obs(self.ctx_s[:, -1])
        return obs, rewards, dones, infos

    def close(self):
        pass

    def get_attr(self, attr_name, indices=None):
        return [getattr(self, attr_name)] * self.num_envs

    def set_attr(self, attr_name, value, indices=None):
        setattr(self, attr_name, value)

    def env_method(self, method_name, *args, indices=None, **kwargs):
        return [getattr(self, method_name)(*args, **kwargs)] * self.num_envs

    def env_is_wrapped(self, wrapper_class, indices=None):
        return [False] * self.num_envs


class EpisodeLogger(BaseCallback):
    """Sammelt (timestep, episode_return) aus den Monitor-Infos."""

    def __init__(self):
        super().__init__()
        self.records = []

    def _on_step(self):
        for info in self.locals["infos"]:
            if "episode" in info:
                self.records.append((self.num_timesteps, info["episode"]["r"]))
        return True

## PPO im gelernten Env trainieren, auf dem echten Pendulum evaluieren

In [ ]:
learned_env = VecNormalize(VecMonitor(LearnedPendulumVecEnv(
    chronos_model, dataset, n_envs=N_ENVS, context_len=CONTEXT_LEN, seed=SEED)))

dyna_ppo = PPO("MlpPolicy", learned_env, n_steps=128, batch_size=512, seed=SEED, verbose=0)
dyna_log = EpisodeLogger()
dyna_ppo.learn(total_timesteps=TOTAL_STEPS, callback=dyna_log, progress_bar=True)

In [ ]:
def evaluate_on_real(ppo_model, obs_rms, n_episodes=50):
    """Echte Episoden-Returns (unnormalisiert) auf Pendulum-v1, Obs-Stats eingefroren."""
    env = VecNormalize(make_vec_env("Pendulum-v1", n_envs=8, seed=SEED + 1000),
                       training=False, norm_obs=True, norm_reward=False)
    env.obs_rms = obs_rms
    returns, _ = evaluate_policy(ppo_model, env, n_eval_episodes=n_episodes,
                                 return_episode_rewards=True)
    env.close()
    return np.asarray(returns)


dyna_returns = evaluate_on_real(dyna_ppo, learned_env.obs_rms)
print(f"Dyna-PPO auf echtem Pendulum: {dyna_returns.mean():.1f} ± {dyna_returns.std():.1f}")

## Baselines: PPO direkt auf dem echten Env (gleiches Budget) und Random

In [ ]:
real_env = VecNormalize(make_vec_env("Pendulum-v1", n_envs=N_ENVS, seed=SEED))
real_ppo = PPO("MlpPolicy", real_env, n_steps=128, batch_size=512, seed=SEED, verbose=0)
real_log = EpisodeLogger()
real_ppo.learn(total_timesteps=TOTAL_STEPS, callback=real_log, progress_bar=True)
real_returns = evaluate_on_real(real_ppo, real_env.obs_rms)
print(f"Real-PPO: {real_returns.mean():.1f} ± {real_returns.std():.1f}")

rng = np.random.default_rng(SEED)
env = gym.make("Pendulum-v1")
random_returns = []
for ep in range(20):
    env.reset(seed=SEED + 2000 + ep)
    random_returns.append(sum(
        env.step(rng.uniform(-2, 2, size=(1,)).astype(np.float32))[1] for _ in range(200)))
env.close()
print(f"Random: {np.mean(random_returns):.1f} ± {np.std(random_returns):.1f} "
      f"('gelöst' ≈ -200)")

In [ ]:
def smooth(records, k=20):
    steps, rets = map(np.array, zip(*records))
    return steps[k - 1:], np.convolve(rets, np.ones(k) / k, mode="valid")


fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(*smooth(dyna_log.records), label="Dyna-PPO (Modell-Steps)")
ax.plot(*smooth(real_log.records), label="Real-PPO (echte Steps)")
ax.axvline(dataset["actions"].size, color="gray", ls=":", label="Offline-Datensatz (real)")
ax.set(xlabel="Trainings-Steps", ylabel="Episoden-Return (geglättet)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

results = {"Dyna-PPO": dyna_returns, "Real-PPO": real_returns, "Random": random_returns}
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(results.keys(), [np.mean(r) for r in results.values()],
       yerr=[np.std(r) for r in results.values()], capsize=4)
ax.set_ylabel("Return auf echtem Pendulum-v1")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()

## Kontrollexperiment: Ist die Pipeline schuld oder das Modell?

Dieselbe Dyna-Pipeline, zwei andere Weltmodelle:

- **wahre Dynamik** (analytische Pendulum-ODE): perfektes Modell → misst die Obergrenze der Pipeline. Erreicht sie Real-PPO-Niveau, ist die Pipeline bewiesen gesund.
- **VARX**: aktions-sensitiv (Notebook 1: Steigung ≈ 0.15), aber linear und mit akkumulierendem Mehrschritt-Fehler.

Befund (Seeds wie oben): wahre Dynamik ≈ **−870** (= Real-PPO −891 → Pipeline gesund), VARX ≈ **−1367** (*schlechter* als Random!). Interpretation: Chronos' Aktions-Blindheit macht die Policy passiv (→ Random-Niveau), während VARX' selbstbewusst-falsche 200-Schritt-Rollouts aktiv ausgebeutet werden (*model exploitation*): die Policy lernt Verhalten, das nur im fehlerhaften Modell gut aussieht. Zwei verschiedene Fehlermodi, beide von Notebook 1 vorhergesagt (Test 2 bzw. Test 4).

In [ ]:
from tsfmrl.pendulum import VARDynamics


class TrueDynamics:
    """Analytische Pendulum-ODE: perfektes Weltmodell (Pipeline-Obergrenze)."""

    name = "wahre Dynamik"

    def predict(self, cs, ca, fa):
        th = cs[:, -1, 0].astype(np.float64)
        thd = cs[:, -1, 1].astype(np.float64)
        preds = np.empty((len(cs), fa.shape[1], 2), dtype=np.float32)
        for h in range(fa.shape[1]):
            thd = np.clip(thd + (15.0 * np.sin(th) + 3.0 * fa[:, h, 0]) * 0.05, -8, 8)
            th = th + thd * 0.05
            preds[:, h, 0], preds[:, h, 1] = th, thd
        return preds


control_returns = {}
for model in [TrueDynamics(), VARDynamics(dataset["states"][:40], dataset["actions"][:40])]:
    env = VecNormalize(VecMonitor(LearnedPendulumVecEnv(
        model, dataset, n_envs=N_ENVS, context_len=CONTEXT_LEN, seed=SEED)))
    ppo = PPO("MlpPolicy", env, n_steps=128, batch_size=512, seed=SEED, verbose=0)
    ppo.learn(total_timesteps=TOTAL_STEPS, progress_bar=True)
    control_returns[model.name] = evaluate_on_real(ppo, env.obs_rms)
    print(f"Dyna-PPO ({model.name}): {control_returns[model.name].mean():.1f} "
          f"± {control_returns[model.name].std():.1f}")

all_results = {"Dyna (Chronos)": dyna_returns, "Dyna (VARX)": list(control_returns.values())[1],
               "Dyna (wahre Dynamik)": list(control_returns.values())[0],
               "Real-PPO": real_returns, "Random": random_returns}
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(all_results.keys(), [np.mean(r) for r in all_results.values()],
       yerr=[np.std(r) for r in all_results.values()], capsize=4)
ax.set_ylabel("Return auf echtem Pendulum-v1")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()